# S4 · AndinaLog 03B · Notebook 1 · Diagnóstico de Inventory Tracking

Este notebook trabaja con el Bronze oficial `andinalog_inventory_tracking.csv`, guardado en `datasets/AndinaLog_03B_Bronce/`. Conserva los doce campos originales y registra problemas sin corregirlos. Las conversiones usadas para validar son temporales. El tratamiento corresponde al notebook 2.

Señala duplicados idénticos y conflictos de `movimiento_id`, fechas vacías, formatos distintos o fechas imposibles, cantidades no numéricas o negativas y contradicciones temporales o cuantitativas. Una salida más merma menor que el ingreso puede representar stock remanente y no se marca como error. En los conflictos de clave marca todas las filas involucradas; no elige un movimiento automáticamente.

Cada ejecución reemplaza cuatro archivos en `proyecto-integrador/andinalog_inventory_tracking/notebook1/salidas/`: el diagnosticado, el detalle de problemas, la cuarentena y el reporte de calidad con SHA-256 del Bronze.

## 1 · Configuración y origen

En local, ejecuta el notebook desde cualquier carpeta dentro del proyecto. En Colab, monta Drive, selecciona `ENTORNO = "drive"` y ajusta `RUTA_PROYECTO_DRIVE` a la carpeta que contiene `datasets/` y `proyecto-integrador/`. Las salidas van a `proyecto-integrador/andinalog_inventory_tracking/notebook1/salidas/` en ese mismo entorno.

In [2]:
from pathlib import Path
import hashlib
import os
import re
import tempfile
import pandas as pd

ENTORNO = "auto"  # "auto", "local" o "drive"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"  # ajustar si la carpeta real es otra
CARPETA_DATASETS = "AndinaLog_03B_Bronce"
NOMBRE_CSV = "andinalog_inventory_tracking.csv"
VERSION_DIAGNOSTICO = "GIAD-M3-S4-INVENTORY-diagnostico-v1"

COLUMNAS_ORIGINALES = [
    "movimiento_id", "lote_id", "producto_id", "centro_distribucion",
    "fecha_ingreso", "fecha_salida", "fecha_vencimiento",
    "cantidad_ingreso", "cantidad_salida", "cantidad_merma",
    "dias_en_almacen", "costo_unitario_bob",
]

def encontrar_raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets" / CARPETA_DATASETS).is_dir() and (carpeta / "proyecto-integrador").is_dir():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz del proyecto; ejecuta dentro de practicasNotebookColab.")

def configurar_rutas(entorno, ruta_drive):
    if entorno == "auto":
        entorno = "drive" if "google.colab" in __import__("sys").modules else "local"
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(ruta_drive)
    elif entorno == "local":
        raiz = encontrar_raiz_local()
    else:
        raise ValueError("ENTORNO debe ser 'auto', 'local' o 'drive'")
    bronze = raiz / "datasets" / CARPETA_DATASETS / NOMBRE_CSV
    salidas = raiz / "proyecto-integrador" / "andinalog_inventory_tracking" / "notebook1" / "salidas"
    if not bronze.is_file():
        raise FileNotFoundError(f"No se encontró el CSV Bronze: {bronze}")
    return bronze, salidas

RUTA_BRONZE, DIRECTORIO_SALIDAS = configurar_rutas(ENTORNO, RUTA_PROYECTO_DRIVE)
print("Bronze:", RUTA_BRONZE)
print("Salidas:", DIRECTORIO_SALIDAS)

Bronze: c:\Users\remrodri\Github\practicasNotebookColab\datasets\AndinaLog_03B_Bronce\andinalog_inventory_tracking.csv
Salidas: c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_inventory_tracking\notebook1\salidas


## 2 · Carga y contrato

La lectura mantiene todas las columnas como texto y los vacíos como cadenas vacías. Las conversiones numéricas y de fecha usadas para comprobar errores son temporales: no se exportan como valores transformados.

In [3]:
def cargar_bronze(ruta):
    huella = hashlib.sha256(ruta.read_bytes()).hexdigest()
    df = pd.read_csv(ruta, dtype="string", encoding="utf-8-sig", keep_default_na=False)
    return df, huella

def validar_esquema(df):
    if list(df.columns) != COLUMNAS_ORIGINALES:
        faltantes = sorted(set(COLUMNAS_ORIGINALES) - set(df.columns))
        extras = sorted(set(df.columns) - set(COLUMNAS_ORIGINALES))
        raise ValueError(f"Esquema inesperado. Faltantes: {faltantes}; extras: {extras}; orden: {list(df.columns)}")
    if not df.columns.is_unique:
        raise ValueError("Hay nombres de columnas duplicados")
    return df

df_bronze, HASH_BRONZE = cargar_bronze(RUTA_BRONZE)
validar_esquema(df_bronze)
print(f"Bronze: {len(df_bronze):,} filas × {len(df_bronze.columns)} columnas")
print("SHA-256:", HASH_BRONZE)
display(df_bronze.head())


Bronze: 6,040 filas × 12 columnas
SHA-256: 4142c8736512c78c553bdd78dd49b6fe304039f7deec8cc096102d31289a1619


,movimiento_id,lote_id,producto_id,centro_distribucion,fecha_ingreso,fecha_salida,fecha_vencimiento,cantidad_ingreso,cantidad_salida,cantidad_merma,dias_en_almacen,costo_unitario_bob
0,MOV-000001,LOT-2026-00001,PROD-006,Santa Cruz,2026-07-09,2026-07-23,2027-07-09,747,743,4,14,29.49
1,MOV-000002,LOT-2026-00002,PROD-031,Tarija,2026-05-21,2026-06-14,2027-05-21,625,615,8,24,50.89
2,MOV-000003,LOT-2026-00003,PROD-021,La Paz,2026-08-18,2026-08-22,2026-09-05,261,252,7,4,83.07
3,MOV-000004,LOT-2026-00004,PROD-009,Cochabamba,2026-07-15,2026-07-30,2026-08-02,97,94,3,15,75.19
4,MOV-000005,LOT-2026-00005,PROD-027,La Paz,2026-06-25,2026-06-28,2026-07-13,83,81,2,3,64.06


## 3 · Catálogo y reglas de diagnóstico

Los códigos identifican el problema exacto para el notebook 2. `DUPLICADO_IDENTICO` marca las copias posteriores de una fila exactamente repetida; `CLAVE_EN_CONFLICTO` marca todos los movimientos con el mismo `movimiento_id` cuando sus contenidos difieren. Un formato de fecha distinto de `YYYY-MM-DD` se separa de una fecha imposible. El formato `DD/MM/YYYY` puede interpretarse temporalmente para comprobar coherencia, sin cambiar el valor Bronze.

`fecha_vencimiento` vacía queda señalada para revisión. El diagnóstico comprueba que salida y vencimiento no sean anteriores al ingreso, que `dias_en_almacen` coincida con la diferencia entre salida e ingreso y que salida más merma no supere ingreso. No considera error un remanente positivo; tampoco presupone que una salida posterior al vencimiento sea incorrecta sin una política operativa aprobada.

In [4]:
CATALOGO_PROBLEMAS = pd.DataFrame([
    *[(c, "FALTANTE", "Identificador vacío") for c in ["movimiento_id", "lote_id", "producto_id"]],
    *[(c, "FORMATO_INVALIDO", "Identificador con formato inesperado") for c in ["movimiento_id", "lote_id", "producto_id"]],
    ("movimiento_id", "DUPLICADO_IDENTICO", "Copia posterior de una fila idéntica"),
    ("movimiento_id", "CLAVE_EN_CONFLICTO", "Movimientos diferentes con la misma clave; se marcan todos"),
    ("centro_distribucion", "FALTANTE", "Centro vacío"),
    ("centro_distribucion", "VALOR_NO_RECONOCIDO", "Centro fuera de los cinco centros observados"),
    *[(c, "FALTANTE", "Fecha vacía") for c in ["fecha_ingreso", "fecha_salida", "fecha_vencimiento"]],
    *[(c, "FORMATO_FECHA_DISTINTO", "Fecha válida con formato diferente de YYYY-MM-DD") for c in ["fecha_ingreso", "fecha_salida", "fecha_vencimiento"]],
    *[(c, "FECHA_INVALIDA", "Fecha imposible o no interpretable") for c in ["fecha_ingreso", "fecha_salida", "fecha_vencimiento"]],
    ("fecha_salida", "ANTERIOR_INGRESO", "Salida anterior al ingreso"),
    ("fecha_vencimiento", "ANTERIOR_INGRESO", "Vencimiento anterior al ingreso"),
    *[(c, "FALTANTE", "Campo numérico vacío") for c in ["cantidad_ingreso", "cantidad_salida", "cantidad_merma", "dias_en_almacen", "costo_unitario_bob"]],
    *[(c, "NO_NUMERICA", "Valor no convertible a número") for c in ["cantidad_ingreso", "cantidad_salida", "cantidad_merma", "dias_en_almacen", "costo_unitario_bob"]],
    *[(c, "NO_ENTERA", "Cantidad o días con fracción") for c in ["cantidad_ingreso", "cantidad_salida", "cantidad_merma", "dias_en_almacen"]],
    *[(c, "NEGATIVA", "Cantidad o días negativos") for c in ["cantidad_ingreso", "cantidad_salida", "cantidad_merma", "dias_en_almacen"]],
    ("cantidad_ingreso", "NO_POSITIVA", "Ingreso igual a cero"),
    ("costo_unitario_bob", "NO_POSITIVO", "Costo cero o negativo"),
    ("cantidad_salida+cantidad_merma", "SUPERA_INGRESO", "Salida más merma supera ingreso"),
    ("dias_en_almacen", "NO_COINCIDE_FECHAS", "Días declarados distintos de salida menos ingreso"),
], columns=["columna_afectada", "codigo_error", "criterio"])
display(CATALOGO_PROBLEMAS)

def registrar_problema(df, mascara, columna, codigo, evidencia=None):
    mascara = mascara.fillna(False).astype(bool)
    filas = df.loc[mascara, ["fila_bronze"]].copy()
    filas["columna_afectada"] = columna
    filas["codigo_error"] = codigo
    if evidencia is None:
        evidencia = df[columna] if columna in df.columns else pd.Series("", index=df.index, dtype="string")
    filas["valor_original"] = evidencia.loc[mascara].astype("string").to_numpy()
    return filas

def texto(df, columna):
    return df[columna].astype("string").str.strip()

def detectar_identificadores(df):
    patrones = {
        "movimiento_id": r"MOV-\d{6}",
        "lote_id": r"LOT-\d{4}-\d{5}",
        "producto_id": r"PROD-\d{3}",
    }
    hallazgos = []
    for col, patron in patrones.items():
        valor = texto(df, col)
        hallazgos.append(registrar_problema(df, valor.eq(""), col, "FALTANTE"))
        hallazgos.append(registrar_problema(df, valor.ne("") & ~valor.str.fullmatch(patron).fillna(False), col, "FORMATO_INVALIDO"))
    return hallazgos

def detectar_duplicados(df):
    clave = texto(df, "movimiento_id")
    firma = df[COLUMNAS_ORIGINALES].astype("string").agg("\x1f".join, axis=1)
    variantes = firma.groupby(clave, dropna=False).transform("nunique")
    conflicto = clave.ne("") & clave.duplicated(keep=False) & variantes.gt(1)
    identico = df[COLUMNAS_ORIGINALES].duplicated(keep="first") & ~conflicto
    return [
        registrar_problema(df, identico, "movimiento_id", "DUPLICADO_IDENTICO"),
        registrar_problema(df, conflicto, "movimiento_id", "CLAVE_EN_CONFLICTO"),
    ]

def detectar_centro(df):
    valor = texto(df, "centro_distribucion")
    conocidos = ["Cochabamba", "Oruro", "Tarija", "La Paz", "Santa Cruz"]
    return [
        registrar_problema(df, valor.eq(""), "centro_distribucion", "FALTANTE"),
        registrar_problema(df, valor.ne("") & ~valor.isin(conocidos), "centro_distribucion", "VALOR_NO_RECONOCIDO"),
    ]

def detectar_fechas(df):
    hallazgos, fechas = [], {}
    for col in ["fecha_ingreso", "fecha_salida", "fecha_vencimiento"]:
        valor = texto(df, col)
        iso = valor.str.fullmatch(r"\d{4}-\d{2}-\d{2}").fillna(False)
        alterno = valor.str.fullmatch(r"\d{2}/\d{2}/\d{4}").fillna(False)
        fecha_iso = pd.to_datetime(valor.where(iso), format="%Y-%m-%d", errors="coerce")
        fecha_alt = pd.to_datetime(valor.where(alterno), format="%d/%m/%Y", errors="coerce")
        fecha = fecha_iso.fillna(fecha_alt)
        fechas[col] = fecha
        hallazgos.extend([
            registrar_problema(df, valor.eq(""), col, "FALTANTE"),
            registrar_problema(df, valor.ne("") & alterno & fecha.notna(), col, "FORMATO_FECHA_DISTINTO"),
            registrar_problema(df, valor.ne("") & fecha.isna(), col, "FECHA_INVALIDA"),
        ])
    ingreso = fechas["fecha_ingreso"]
    for col in ["fecha_salida", "fecha_vencimiento"]:
        fecha = fechas[col]
        hallazgos.append(registrar_problema(df, ingreso.notna() & fecha.notna() & fecha.lt(ingreso), col, "ANTERIOR_INGRESO"))
    return hallazgos, fechas

def detectar_medicion(df, columna, entero=False, positivo=False):
    valor = texto(df, columna)
    numero = pd.to_numeric(valor, errors="coerce")
    hallazgos = [
        registrar_problema(df, valor.eq(""), columna, "FALTANTE"),
        registrar_problema(df, valor.ne("") & numero.isna(), columna, "NO_NUMERICA"),
    ]
    if entero:
        hallazgos.extend([
            registrar_problema(df, numero.notna() & numero.mod(1).ne(0), columna, "NO_ENTERA"),
            registrar_problema(df, numero.notna() & numero.lt(0), columna, "NEGATIVA"),
        ])
    if positivo:
        codigo = "NO_POSITIVA" if columna == "cantidad_ingreso" else "NO_POSITIVO"
        hallazgos.append(registrar_problema(df, numero.notna() & numero.eq(0) if entero else numero.notna() & numero.le(0), columna, codigo))
    return hallazgos, numero

def detectar_coherencia(df, fechas, numeros):
    ingreso, salida, merma = (numeros[c] for c in ["cantidad_ingreso", "cantidad_salida", "cantidad_merma"])
    comparables = ingreso.notna() & salida.notna() & merma.notna() & ingreso.ge(0) & salida.ge(0) & merma.ge(0)
    exceso = comparables & (salida + merma).gt(ingreso)
    evidencia = texto(df, "cantidad_salida") + " + " + texto(df, "cantidad_merma") + " > " + texto(df, "cantidad_ingreso")
    fecha_ingreso, fecha_salida = fechas["fecha_ingreso"], fechas["fecha_salida"]
    dias = numeros["dias_en_almacen"]
    diferencia = (fecha_salida - fecha_ingreso).dt.days
    dias_incoherentes = fecha_ingreso.notna() & fecha_salida.notna() & dias.notna() & dias.ne(diferencia)
    return [
        registrar_problema(df, exceso, "cantidad_salida+cantidad_merma", "SUPERA_INGRESO", evidencia),
        registrar_problema(df, dias_incoherentes, "dias_en_almacen", "NO_COINCIDE_FECHAS"),
    ]

def diagnosticar(df_bronze):
    principal = df_bronze.copy(deep=True)
    principal.insert(0, "fila_bronze", range(1, len(principal) + 1))
    hallazgos = detectar_identificadores(principal) + detectar_duplicados(principal) + detectar_centro(principal)
    hallazgos_fechas, fechas = detectar_fechas(principal)
    hallazgos += hallazgos_fechas
    numeros = {}
    for col in ["cantidad_ingreso", "cantidad_salida", "cantidad_merma", "dias_en_almacen", "costo_unitario_bob"]:
        nuevos, numeros[col] = detectar_medicion(principal, col, entero=col != "costo_unitario_bob", positivo=col in ["cantidad_ingreso", "costo_unitario_bob"])
        hallazgos += nuevos
    hallazgos += detectar_coherencia(principal, fechas, numeros)
    problemas = pd.concat(hallazgos, ignore_index=True)
    problemas = problemas.sort_values(["fila_bronze", "columna_afectada", "codigo_error"], kind="stable").reset_index(drop=True)
    problemas["version_diagnostico"] = VERSION_DIAGNOSTICO
    columnas_por_fila = problemas.groupby("fila_bronze")["columna_afectada"].agg(
        lambda valores: "|".join(dict.fromkeys(valores))
    )
    principal["columnas_con_problemas"] = principal["fila_bronze"].map(columnas_por_fila).fillna("")
    principal["en_cuarentena"] = principal["columnas_con_problemas"].ne("")
    return principal, problemas

df_diagnosticado, df_problemas = diagnosticar(df_bronze)
df_cuarentena = df_diagnosticado.loc[df_diagnosticado["en_cuarentena"]].copy()
print(f"Principal: {len(df_diagnosticado):,}; problemas: {len(df_problemas):,}; filas en cuarentena: {len(df_cuarentena):,}")
display(df_problemas.groupby(["columna_afectada", "codigo_error"]).size().rename("filas").reset_index())

,columna_afectada,codigo_error,criterio
0,movimiento_id,FALTANTE,Identificador vacío
1,lote_id,FALTANTE,Identificador vacío
2,producto_id,FALTANTE,Identificador vacío
3,movimiento_id,FORMATO_INVALIDO,Identificador con formato inesperado
4,lote_id,FORMATO_INVALIDO,Identificador con formato inesperado
5,producto_id,FORMATO_INVALIDO,Identificador con formato inesperado
6,movimiento_id,DUPLICADO_IDENTICO,Copia posterior de una fila idéntica
7,movimiento_id,CLAVE_EN_CONFLICTO,Movimientos diferentes con la misma clave; se ...
8,centro_distribucion,FALTANTE,Centro vacío
9,centro_distribucion,VALOR_NO_RECONOCIDO,Centro fuera de los cinco centros observados


Principal: 6,040; problemas: 134; filas en cuarentena: 133


,columna_afectada,codigo_error,filas
0,cantidad_ingreso,NO_NUMERICA,10
1,cantidad_merma,NEGATIVA,5
2,fecha_ingreso,FORMATO_FECHA_DISTINTO,25
3,fecha_salida,FECHA_INVALIDA,5
4,fecha_vencimiento,FALTANTE,18
5,movimiento_id,CLAVE_EN_CONFLICTO,2
6,movimiento_id,DUPLICADO_IDENTICO,39
7,producto_id,FORMATO_INVALIDO,30


## 4 · Reporte y comprobaciones antes de exportar

El reporte registra la huella SHA-256 para reconocer la versión exacta del CSV de origen. Los conteos de problemas pueden superar el número de filas en cuarentena porque una fila puede tener varios hallazgos. Las métricas representan hallazgos, no decisiones de corrección.

In [5]:
def construir_reporte(df_bronze, principal, problemas, ruta, huella):
    conteos = problemas.groupby(["columna_afectada", "codigo_error"]).size()
    datos = [
        ("archivo_bronze", ruta.name),
        ("sha256_bronze", huella),
        ("version_diagnostico", VERSION_DIAGNOSTICO),
        ("filas_bronze", len(df_bronze)),
        ("filas_diagnosticadas", len(principal)),
        ("filas_en_cuarentena", int(principal["en_cuarentena"].sum())),
        ("filas_sin_cuarentena", int((~principal["en_cuarentena"]).sum())),
        ("problemas_detectados", len(problemas)),
    ]
    datos += [(f"{col}:{codigo}", int(total)) for (col, codigo), total in conteos.items()]
    return pd.DataFrame(datos, columns=["metrica", "valor"])

def validar_resultados(df_bronze, principal, problemas, cuarentena, reporte):
    assert list(principal.columns) == ["fila_bronze", *COLUMNAS_ORIGINALES, "columnas_con_problemas", "en_cuarentena"]
    pd.testing.assert_frame_equal(principal[COLUMNAS_ORIGINALES], df_bronze[COLUMNAS_ORIGINALES])
    assert len(principal) == len(df_bronze)
    assert principal["fila_bronze"].is_unique
    assert len(cuarentena) == int(principal["en_cuarentena"].sum())
    assert problemas["fila_bronze"].isin(principal["fila_bronze"]).all()
    assert problemas[["columna_afectada", "codigo_error"]].apply(tuple, axis=1).isin(
        CATALOGO_PROBLEMAS[["columna_afectada", "codigo_error"]].apply(tuple, axis=1)
    ).all()
    assert set(problemas["fila_bronze"]) == set(cuarentena["fila_bronze"])
    assert len(reporte) >= 8

reporte_calidad = construir_reporte(df_bronze, df_diagnosticado, df_problemas, RUTA_BRONZE, HASH_BRONZE)
validar_resultados(df_bronze, df_diagnosticado, df_problemas, df_cuarentena, reporte_calidad)
display(reporte_calidad)
print("Comprobaciones previas a la exportación: correctas")


,metrica,valor
0,archivo_bronze,andinalog_inventory_tracking.csv
1,sha256_bronze,4142c8736512c78c553bdd78dd49b6fe304039f7deec8c...
2,version_diagnostico,GIAD-M3-S4-INVENTORY-diagnostico-v1
3,filas_bronze,6040
4,filas_diagnosticadas,6040
5,filas_en_cuarentena,133
6,filas_sin_cuarentena,5907
7,problemas_detectados,134
8,cantidad_ingreso:NO_NUMERICA,10
9,cantidad_merma:NEGATIVA,5


Comprobaciones previas a la exportación: correctas


## 5 · Exportación reproducible

Los cuatro CSV se escriben primero como archivos temporales en `proyecto-integrador/andinalog_inventory_tracking/notebook1/salidas/` y se reemplazan con el mismo nombre al final. El CSV Bronze nunca se sobrescribe. Si se vuelve a ejecutar con la misma fuente y reglas, las salidas se actualizan en lugar de acumular versiones antiguas.

In [6]:
def exportar_salidas(directorio, tablas, ruta_bronze, huella_inicial):
    if hashlib.sha256(ruta_bronze.read_bytes()).hexdigest() != huella_inicial:
        raise RuntimeError("El CSV Bronze cambió durante la ejecución; no se exportarán resultados")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_inventory_", dir=directorio,
                                             encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

tablas_salida = {
    "andinalog_inventory_tracking_diagnosticado.csv": df_diagnosticado,
    "andinalog_inventory_tracking_problemas.csv": df_problemas,
    "andinalog_inventory_tracking_cuarentena.csv": df_cuarentena,
    "andinalog_inventory_tracking_reporte_calidad.csv": reporte_calidad,
}
rutas_creadas = exportar_salidas(DIRECTORIO_SALIDAS, tablas_salida, RUTA_BRONZE, HASH_BRONZE)
for ruta in rutas_creadas:
    print(ruta)
print("Bronze intacta; salidas anteriores reemplazadas")

c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_inventory_tracking\notebook1\salidas\andinalog_inventory_tracking_diagnosticado.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_inventory_tracking\notebook1\salidas\andinalog_inventory_tracking_problemas.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_inventory_tracking\notebook1\salidas\andinalog_inventory_tracking_cuarentena.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_inventory_tracking\notebook1\salidas\andinalog_inventory_tracking_reporte_calidad.csv
Bronze intacta; salidas anteriores reemplazadas


## Siguiente etapa

El notebook 2 leerá el archivo diagnosticado y el detalle de problemas. El informe de S4 justifica tratamientos posibles, pero ninguna regla de curación está aprobada automáticamente por este diagnóstico. Una fila saldrá de cuarentena solo cuando todos sus problemas hayan sido resueltos y validados.